In [3]:
#Install libraries

!pip install transformers datasets torch scikit-learn evaluate accelerate -q

In [4]:
#Import libraries

import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import accuracy_score, f1_score

In [6]:
# Load the AG News dataset from Hugging Face
dataset = load_dataset("fancyzhx/ag_news")

# Quick check of the dataset structure
print(dataset)
print("Example:", dataset["train"][0])

# Labels: 0=World, 1=Sports, 2=Business, 3=Sci/Tech
label_names = dataset["train"].features["label"].names
num_labels = len(label_names)
print("Labels:", label_names)

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})
Example: {'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}
Labels: ['World', 'Sports', 'Business', 'Sci/Tech']


In [7]:
MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    # Pad/truncate every text to a fixed length (128 tokens)
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )

# Apply tokenization to the whole dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Using a smaller subset for faster training on Colab.
# Remove the .select(...) part if you want to train on the full dataset.
train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(8000))
test_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(2000))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [8]:
# Load pretrained BERT with a classification head for 4 classes
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
def compute_metrics(eval_pred):
    # Called automatically by Trainer during evaluation
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)   # convert logits to predicted class
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="weighted")
    return {"accuracy": accuracy, "f1_score": f1}

In [10]:
training_args = TrainingArguments(
    output_dir="./bert_agnews_model",   # where checkpoints are saved
    eval_strategy="epoch",              # evaluate after every epoch
    save_strategy="epoch",              # save model after every epoch
    learning_rate=2e-5,                 # standard LR for BERT fine-tuning
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,                 # 3 epochs is usually enough for good results
    weight_decay=0.01,                  # helps reduce overfitting
    logging_dir="./logs",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_score",
    report_to="none",                   # disable wandb/tensorboard logging
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [11]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

# Fine-tune the model
# (Enable GPU first: Runtime > Change runtime type > T4 GPU)
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Score
1,0.276564,0.297457,0.907000,0.906944
2,0.254817,0.286484,0.916000,0.915980
3,0.100175,0.315180,0.917500,0.917400


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=1500, training_loss=0.24373639774322509, metrics={'train_runtime': 644.0666, 'train_samples_per_second': 37.263, 'train_steps_per_second': 2.329, 'total_flos': 1578694680576000.0, 'train_loss': 0.24373639774322509, 'epoch': 3.0})

In [12]:
# Evaluate the fine-tuned model on the test set
results = trainer.evaluate()
print("Accuracy:", results["eval_accuracy"])
print("F1-score:", results["eval_f1_score"])

Training Loss,Validation Loss,Epoch,Accuracy,F1 Score
0.100175,0.315180,3,0.917500,0.917400


Accuracy: 0.9175
F1-score: 0.9173996557811607


In [13]:
# Save the fine-tuned model and tokenizer for later use (e.g. deployment)
SAVE_PATH = "./final_bert_agnews_model"
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print("Model saved at:", SAVE_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved at: ./final_bert_agnews_model


In [14]:
!pip install gradio -q

In [15]:
import gradio as gr
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SAVE_PATH = "./final_bert_agnews_model"
LABELS = ["World", "Sports", "Business", "Sci/Tech"]

# Load the fine-tuned model and tokenizer we saved in Cell 10
tokenizer = AutoTokenizer.from_pretrained(SAVE_PATH)
model = AutoModelForSequenceClassification.from_pretrained(SAVE_PATH)
model.eval()   # inference mode

def predict_topic(text):
    # Tokenize the input text the same way we did during training
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=128,
    )

    # No gradient calculation needed for inference (faster)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1).squeeze().tolist()

    # Build a dictionary of {label: probability} for Gradio's Label output
    result = {LABELS[i]: float(probs[i]) for i in range(len(LABELS))}
    return result

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [16]:
# Build a simple web UI with a text box as input and probability scores as output
demo = gr.Interface(
    fn=predict_topic,
    inputs=gr.Textbox(
        lines=3,
        placeholder="e.g. Apple unveils new AI chip for next-gen iPhones",
        label="Enter a news headline or paragraph",
    ),
    outputs=gr.Label(num_top_classes=4, label="Predicted Topic"),
    title="📰 News Topic Classifier (BERT)",
    description="Fine-tuned bert-base-uncased model trained on the AG News dataset. Classifies text into World, Sports, Business, or Sci/Tech.",
)

# share=True gives a public link too (useful for showing to others)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8e5e0142b4a31d188b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
